# FloodEye: huấn luyện trên Kaggle

Notebook chỉ làm ba việc: lấy mã ở một commit cố định, chuẩn bị dữ liệu bằng script của kho, gọi `training/train.py`. Mọi logic nằm trong kho mã.

Trước khi chạy: bật **GPU T4** và **Internet** ở khung Settings (Internet cần tài khoản đã xác minh số điện thoại).

In [ ]:
COMMIT = "main"  # đổi thành mã commit đầy đủ trước lần chạy lấy số liệu
CONFIG = "configs/experiments/E2_s0.yaml"

!git clone -q https://github.com/huzy179/uav-rescue-qa.git /kaggle/working/floodeye
%cd /kaggle/working/floodeye
!git checkout -q {COMMIT} && git log -1 --oneline
!pip install -q -e ".[dev]"

## Dữ liệu

VisDrone train và val, chuyển về một lớp `person`.

In [ ]:
%%bash
set -e
mkdir -p data/raw/visdrone && cd data/raw/visdrone
for split in train val; do
  curl -sLO https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-$split.zip
  unzip -q VisDrone2019-DET-$split.zip && rm VisDrone2019-DET-$split.zip
done
cd ../../..
for split in train val; do
  python tools/convert_visdrone.py --src data/raw/visdrone/VisDrone2019-DET-$split --split $split
done

## Huấn luyện

Phiên bị ngắt thì chạy lại ô dưới với `--resume` thêm vào cuối lệnh, sau khi đã chép `runs/train/` của phiên trước về đúng chỗ.

In [ ]:
!python training/train.py --config {CONFIG} --device 0

## Dọn dẹp

Xoá dữ liệu để phần Output của notebook chỉ còn mã và `runs/train/` (trọng số, `results.csv`, `run_config.json`). Lưu Output thành Kaggle Dataset có phiên bản rồi tải về `runs/train/` trên máy.

In [ ]:
!rm -rf data